# `yellow_tripdata_2025-07.parquet` → taxi-zone/hour demand CSV

Rather than a raw row-for-row dump, this aggregates trips into one row per
**(year, taxi zone, month, weekday, hour)** — the shape needed by this
project's `historical_taxi_demand` table
(`database/migrations/012_create_historical_taxi_demand.sql`):

| column | meaning |
|---|---|
| `source_year` | calendar year of the trip |
| `taxi_zone_id` | **drop-off** zone (`DOLocationID`) — dropoffs near a restaurant are the foot-traffic signal this feeds |
| `month` | 1–12 |
| `weekday` | pandas convention: Monday=0 … Sunday=6 |
| `hour` | 0–23 |
| `dropoff_count` | number of trips dropped off in that zone/hour |
| `passenger_count_sum` | total passengers across those trips |
| `avg_trip_distance` | mean trip distance (miles); null if the group has no valid distances |

All time fields are derived from `tpep_dropoff_datetime`, matching `taxi_zone_id`'s use of the drop-off location — both describe the same event.

## Step 1 — Install dependencies

In [ ]:
%pip install pyarrow pandas

## Step 2 — Imports and file paths

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd

PARQUET_PATH = Path.home() / "Downloads" / "yellow_tripdata_2025-07.parquet"
CSV_PATH = PARQUET_PATH.with_name(PARQUET_PATH.stem + "_taxi_zone_hourly.csv")

print(PARQUET_PATH, "exists:", PARQUET_PATH.exists())

## Step 3 — Inspect the file before loading it

`pq.ParquetFile` reads just the metadata (row count, schema) without pulling any data into memory — worth doing first on any file you haven't seen before.

In [ ]:
pf = pq.ParquetFile(PARQUET_PATH)

print(f"Rows:    {pf.metadata.num_rows:,}")
print(f"Columns: {pf.schema_arrow.names}")

## Step 4 — Load only the columns the aggregation needs

The source file has 20 columns; the target CSV only needs 3
(`tpep_dropoff_datetime`, `DOLocationID`, `passenger_count`, `trip_distance`).
Passing `columns=` to `read_parquet` skips reading the other 16/20 columns
entirely — Parquet is columnar, so this is much cheaper than loading
everything and dropping columns afterward.

In [ ]:
df = pd.read_parquet(
    PARQUET_PATH,
    columns=["tpep_dropoff_datetime", "DOLocationID", "passenger_count", "trip_distance"],
)
df.head()

## Step 4b — Filter to July 2025 only

Every monthly TLC file has a handful of mistimed records outside the nominal month (a known data-quality quirk, not a bug). For this file: 3,897,746 rows are genuinely 2025-07, plus 1,214 in 2025-08, 2 in 2025-06, and 1 in 2009-01. Dropping the stray rows here so the output is strictly July 2025.

In [ ]:
before = len(df)
df = df[
    (df["tpep_dropoff_datetime"].dt.year == 2025)
    & (df["tpep_dropoff_datetime"].dt.month == 7)
].copy()

print(f"Dropped {before - len(df):,} stray rows outside 2025-07 ({len(df):,} remain)")

## Step 5 — Derive the time fields

`tpep_dropoff_datetime` is already a `datetime64` dtype once read from Parquet (no manual parsing needed), so `.dt` accessors work directly.

In [ ]:
df["source_year"] = df["tpep_dropoff_datetime"].dt.year
df["month"] = df["tpep_dropoff_datetime"].dt.month
df["weekday"] = df["tpep_dropoff_datetime"].dt.weekday  # Monday=0 ... Sunday=6
df["hour"] = df["tpep_dropoff_datetime"].dt.hour
df["taxi_zone_id"] = df["DOLocationID"]

df[["tpep_dropoff_datetime", "source_year", "month", "weekday", "hour", "taxi_zone_id"]].head()

## Step 6 — Aggregate: one row per (year, zone, month, weekday, hour)

- `dropoff_count` — `size` counts every row in the group, including rows with missing passenger/distance values.
- `passenger_count_sum` — `sum` skips NaNs by default, so missing passenger counts contribute 0 rather than breaking the sum.
- `avg_trip_distance` — `mean` also skips NaNs; if a group has *no* valid distances at all, this naturally comes out as `NaN`, which becomes an empty/null cell in the CSV (matching the target column's nullable definition).

In [ ]:
agg = (
    df.groupby(["source_year", "taxi_zone_id", "month", "weekday", "hour"])
    .agg(
        dropoff_count=("taxi_zone_id", "size"),
        passenger_count_sum=("passenger_count", "sum"),
        avg_trip_distance=("trip_distance", "mean"),
    )
    .reset_index()
)

# Keep the exact column order requested
agg = agg[[
    "source_year", "taxi_zone_id", "month", "weekday", "hour",
    "dropoff_count", "passenger_count_sum", "avg_trip_distance",
]]

# Tidy dtypes: whole-number fields as int, distance rounded for readability
agg["passenger_count_sum"] = agg["passenger_count_sum"].fillna(0).astype("int64")
agg["avg_trip_distance"] = agg["avg_trip_distance"].round(3)

print(f"{len(df):,} trips -> {len(agg):,} aggregated (year, zone, month, weekday, hour) rows")
agg.head(10)

## Step 7 — Export to CSV

In [ ]:
agg.to_csv(CSV_PATH, index=False)

size_kb = CSV_PATH.stat().st_size / 1024
print(f"Wrote {CSV_PATH} ({size_kb:,.1f} KB, {len(agg):,} rows)")

## Step 8 (optional) — Chunked aggregation for much larger inputs

Not needed for a single month (3.9M rows loads fine directly), but if you
later run this across many months/years combined, loading everything at
once may not fit in memory. The pattern: aggregate each batch separately,
then **combine by summing counts/sums — never by averaging averages** (that
would weight each batch equally regardless of size, which is wrong). The
true mean is recovered at the end as `distance_sum / distance_count`.

In [ ]:
GROUP_COLS = ["source_year", "taxi_zone_id", "month", "weekday", "hour"]

pf = pq.ParquetFile(PARQUET_PATH)
partials = []

for batch in pf.iter_batches(
    batch_size=500_000,
    columns=["tpep_dropoff_datetime", "DOLocationID", "passenger_count", "trip_distance"],
):
    chunk = batch.to_pandas()
    chunk = chunk[
        (chunk["tpep_dropoff_datetime"].dt.year == 2025)
        & (chunk["tpep_dropoff_datetime"].dt.month == 7)
    ]
    chunk["source_year"] = chunk["tpep_dropoff_datetime"].dt.year
    chunk["month"] = chunk["tpep_dropoff_datetime"].dt.month
    chunk["weekday"] = chunk["tpep_dropoff_datetime"].dt.weekday
    chunk["hour"] = chunk["tpep_dropoff_datetime"].dt.hour
    chunk["taxi_zone_id"] = chunk["DOLocationID"]

    partial = (
        chunk.groupby(GROUP_COLS)
        .agg(
            dropoff_count=("taxi_zone_id", "size"),
            passenger_count_sum=("passenger_count", "sum"),
            distance_sum=("trip_distance", "sum"),
            distance_count=("trip_distance", "count"),
        )
        .reset_index()
    )
    partials.append(partial)

combined = (
    pd.concat(partials)
    .groupby(GROUP_COLS)
    .sum()  # sums the per-batch counts/sums together
    .reset_index()
)
combined["avg_trip_distance"] = (
    combined["distance_sum"] / combined["distance_count"].replace(0, pd.NA)
).round(3)

chunked_agg = combined[[
    "source_year", "taxi_zone_id", "month", "weekday", "hour",
    "dropoff_count", "passenger_count_sum", "avg_trip_distance",
]]

# Sanity check: should match the direct (non-chunked, filtered) result exactly
print("Matches direct aggregation:", chunked_agg.equals(agg))
chunked_agg.head(10)